In [4]:
import cv2
import os
import numpy as np
import csv
from skimage.metrics import structural_similarity as ssim  # ✅ already installed

frames_folder = "frames"
gaussian_folder = "denoised_gaussian"
median_folder = "denoised_median"

def psnr(img1, img2):
    img1 = img1.astype(np.float64)  # ✅ float fix
    img2 = img2.astype(np.float64)  # ✅ float fix
    mse_val = np.mean((img1 - img2) ** 2)
    if mse_val == 0:
        return float('inf')
    return 20 * np.log10(255.0 / np.sqrt(mse_val))

def mse(img1, img2):
    img1 = img1.astype(np.float64)  # ✅ float fix
    img2 = img2.astype(np.float64)  # ✅ float fix
    return np.mean((img1 - img2) ** 2)

def compute_ssim(img1, img2):       # ✅ added ssim
    return round(ssim(img1, img2, data_range=255), 4)

frame_files = sorted([f for f in os.listdir(frames_folder) if f.endswith(".png")])
results = []

for idx, file in enumerate(frame_files[:30]):
    frame_number = file.split("_")[1].split(".")[0]

    original = cv2.imread(os.path.join(frames_folder, file), cv2.IMREAD_GRAYSCALE)
    gaussian = cv2.imread(os.path.join(gaussian_folder, file), cv2.IMREAD_GRAYSCALE)
    median   = cv2.imread(os.path.join(median_folder, file), cv2.IMREAD_GRAYSCALE)

    # ✅ resize fix — makes all images same size before comparing
    original = cv2.resize(original, (640, 480))
    gaussian = cv2.resize(gaussian, (640, 480))
    median   = cv2.resize(median,   (640, 480))

    results.append({
        "Frame"         : frame_number,
        "PSNR_Gaussian" : round(psnr(original, gaussian), 2),
        "PSNR_Median"   : round(psnr(original, median), 2),
        "MSE_Gaussian"  : round(mse(original, gaussian), 2),
        "MSE_Median"    : round(mse(original, median), 2),
        "SSIM_Gaussian" : compute_ssim(original, gaussian),
        "SSIM_Median"   : compute_ssim(original, median),
    })

csv_file = "denoising_metrics.csv"
with open(csv_file, mode='w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)

print(f"✅ PSNR, MSE & SSIM metrics for 30 frames saved in '{csv_file}'")

✅ PSNR, MSE & SSIM metrics for 30 frames saved in 'denoising_metrics.csv'
